In [1]:
import pandas as pd
import numpy as np
import ingestion_db as ig
from sqlalchemy import create_engine


In [13]:
#To check if the CSV data is loaded into DB
ig.load_raw_data()
conn = create_engine("sqlite:///traffic.db")

fines = pd.read_sql("SELECT * FROM Fines", conn)

fines = fines.iloc[:-1, :]

fines.to_sql("Fines", conn, if_exists="replace", index=False)

91

In [14]:
#averages of all the column



fines_avgs = pd.read_sql_query("""
    SELECT
        ROUND(AVG(Total_Challan), 2) AS avg_challan,
        ROUND(AVG(paid_Challan), 2) AS avg_paid_challan,
        ROUND(
            SUM(paid_Challan) * 100.0 /
            NULLIF(SUM(Total_Challan), 0),
            2
        ) AS overall_paid_challan_rate,
        ROUND(AVG(Total_Challan_Amount), 2) AS avg_challan_amount,
        ROUND(AVG(Unpaid_Challan_Amount), 2) AS avg_unpaid_challan_amount,
        ROUND(
            SUM(Unpaid_Challan_Amount) * 100.0 /
            NULLIF(SUM(Total_Challan_Amount), 0),
            2
        ) AS overall_unpaid_challan_amount_rate
    FROM fines
""", conn)

fines_avgs

ig.ingest_db(fines_avgs, "fines_avgs", conn)
fines_avgs

,avg_challan,avg_paid_challan,overall_paid_challan_rate,avg_challan_amount,avg_unpaid_challan_amount,overall_unpaid_challan_amount_rate
0,9316.11,1680.16,18.04,7371125.27,6048625.82,82.06


In [ ]:
#What share of challans and rupees goes unpaid, citywide and per station?


pd.read_sql("with fines_rank as (SELECT *,row_number() over (order by Total_Challan_Amount DESC) as ranking from fines)" \
"SELECT Traffic_Police_Station, round((Unpaid_Challan*100.0/Total_Challan),2) as Unpaid_rate_across_city," \
" round(Unpaid_Challan_Amount*100.0/Total_Challan_Amount,2) as Unpaid_Amount_Rate_Across_City FROM fines_rank " \
"WHERE ranking > 1",conn)
    

,Traffic_Police_Station,Unpaid_rate_across_city,Unpaid_Amount_Rate_Across_City
0,N.M.Joshi Police Station,74.53,72.02
1,Khar Police Station,85.65,86.63
2,Andheri Police Station,84.63,86.23
3,Borivali Police Station,79.35,80.74
4,Dharavi Police Station,75.49,78.19
...,...,...,...
85,B.K.C Police Station,81.93,83.34
86,M.H.B Police Station,85.02,85.12
87,Azad maidan Police Station,73.59,71.33
88,Marin Drive Police Station,78.18,78.47


In [5]:
# Are unpaid tickets bigger fines than paid ones? (average rupees per paid vs unpaid ticket)

pd.read_sql("with fines_rank as (SELECT *,row_number() over (order by Total_Challan_Amount DESC) as ranking from fines)," \
"final_fines as (SELECT Traffic_Police_Station,  round(Paid_Challan_Amount*1.0/NULLIF(Paid_Challan, 0),2) as avg_fine_per_paid_challan," \
"round(Unpaid_Challan_Amount*1.0/NULLIF(UnPaid_Challan, 0),2) as avg_fine_per_unpaid_challan FROM fines_rank " \
"WHERE ranking > 1)" \
"SELECT *, avg_fine_per_unpaid_challan - avg_fine_per_paid_challan as difference from final_fines ",conn)



,Traffic_Police_Station,avg_fine_per_paid_challan,avg_fine_per_unpaid_challan,difference
0,Malwani Police Station,735.82,750.41,14.59
1,N.M.Joshi Police Station,959.70,844.19,-115.51
2,Khar Police Station,674.39,731.92,57.53
3,Andheri Police Station,726.36,826.06,99.70
4,Borivali Police Station,789.51,861.04,71.53
...,...,...,...,...
86,B.K.C Police Station,813.16,896.81,83.65
87,M.H.B Police Station,959.88,967.83,7.95
88,Azad maidan Police Station,950.00,848.22,-101.78
89,Marin Drive Police Station,754.17,766.86,12.69


In [28]:
#Where is the unpaid money concentrated? How much of the total sits in the top 10 stations?

# What are top 10 highest unpaid police station
top_10_unpaid = pd.read_sql("with top_unpaid as(SELECT Traffic_Police_Station," \
"Unpaid_Challan," \
"Unpaid_Challan_Amount, " \
"ROW_NUMBER() OVER (ORDER BY Unpaid_Challan_Amount DESC) as ranking  FROM fines)" \
"SELECT Traffic_Police_Station,Unpaid_Challan,Unpaid_Challan_Amount from top_unpaid WHERE ranking Between 2 AND 11 ",conn)



top_10_unpaid

,Traffic_Police_Station,Unpaid_Challan,Unpaid_Challan_Amount
0,Khar Police Station,33080,24211800
1,Andheri Police Station,25436,21011650
2,N.M.Joshi Police Station,24576,20746850
3,Borivali Police Station,21468,18484850
4,Mulund Police Station,21452,16151650
5,Ghatkopar Police Station,20157,15238700
6,Dharavi Police Station,21773,14860750
7,Pydhonie Police Station,17457,14814150
8,R.A.K Marg Police Station,19904,14535400
9,Nagpada Police Station,15796,13788100


In [29]:
#what % of top 10 unpaid contributes to overall unpaid
unpaid_top_10_rate = pd.read_sql("with top_unpaid as(SELECT Traffic_Police_Station," \
"Unpaid_Challan," \
"Unpaid_Challan_Amount," \
"SUM(Total_Challan_Amount) over () as Station_Total," \
"ROW_NUMBER() OVER (ORDER BY Unpaid_Challan_Amount DESC) as ranking  FROM fines)" \
"SELECT round(sum(Unpaid_Challan_Amount)*100.0/Station_Total,2) as top_10_rate from top_unpaid WHERE ranking Between 2 AND 11",conn)


unpaid_top_10_rate

,top_10_rate
0,25.92


In [17]:

#Is non-payment uniform, or do smaller stations collect better than larger ones?

fin_fines = fines[["Traffic_Police_Station","Total_Challan","Unpaid_Challan"]]

fin_fines["unpaid_rate"] = round((fin_fines["Unpaid_Challan"]/fin_fines["Total_Challan"])*100,2)
fin_fines["unpaid_rate"].describe()

fin_fines["station_size"] = pd.qcut(
    fines["Total_Challan"].rank(method="first"),
    q=3,
    labels=["Small", "Medium", "Large"]
)

fin_fines.groupby("station_size").agg(
    number_of_station = ("Traffic_Police_Station","count"),
    avg_total_challan = ("Total_Challan","mean"),
    avg_unpaid_rate = ("unpaid_rate","mean"),
    median_unpaid_rate = ("unpaid_rate","median")
).round(2).reset_index()

C:\Users\HP\AppData\Local\Temp\ipykernel_13460\422734192.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fin_fines["unpaid_rate"] = round((fin_fines["Unpaid_Challan"]/fin_fines["Total_Challan"])*100,2)
C:\Users\HP\AppData\Local\Temp\ipykernel_13460\422734192.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fin_fines["station_size"] = pd.qcut(
C:\Users\HP\AppData\Local\Temp\ipykernel_13460\422734192.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a futu

,station_size,number_of_station,avg_total_challan,avg_unpaid_rate,median_unpaid_rate
0,Small,30,1823.27,80.60,81.78
1,Medium,30,6353.57,80.88,81.40
2,Large,31,19434.23,82.10,82.74


In [ ]:
#Which violation types dominate citywide?

df = pd.read_csv("violation.csv")

df.index = df.iloc[:,:1]
df.index = df.index.str[0]
df = df.drop(columns=["Traffic_Police_Station"])
df.astype(int)

df = df.iloc[:-1,:-1]

df["highest_violation_count"] = df.max(axis=1)
df["highest_violation"] = df.idxmax(axis=1)

high = df["highest_violation"].reset_index()
highest_violation_across_city = high.groupby("highest_violation").count().reset_index().rename(columns={"index":"total"}).sort_values("total", ascending=False).head(1)
highest_violation_across_city
#Without helmet dominates the city with across 60/91 police stations recording highest amount of cases for without helmet

,highest_violation,total
2,Without_Helmet,60


In [19]:
#Do stations differ in their dominant violation?

high.groupby("highest_violation").count().reset_index().rename(columns={"index":"total"})

#there are 3 different type of violation where maximum cases has been resgistered 

,highest_violation,total
0,Causing_Obstruction_To_Free_Flow_Of_Traffic,10
1,Other_MV_Acts,21
2,Without_Helmet,60


In [ ]:
#Is enforcement weighted toward dangerous violations?

High_risk_col = ['Drunk_and_Drive','Speed_Violation','Without_Seatbelt','Wrong_side_driving','Lane_cutting', 'Jumping_signal','Without_Helmet']
Low_risk_col = ['Carry_excess_Passenger', 'Causing_Obstruction_To_Free_Flow_Of_Traffic',
                'Demanding_excess_fare_from_the_passenger',
                'Disobedience_of_Police_order/direction','Failed_to_produce_valid_PUCC', 'Honking',
                'Modification_in_Silencer','No_Entry_In_Oneway','Other_MV_Acts','Police_stop_violation',
                'Refusal_To_Ply','Tinted_glasses_Black_Film','Triple_Seat','Unauthorised_Modification_In_Vehicle',
                'WO_Permit','Without_MDL','Without_Uniform']





total_high_risk = df[High_risk_col].sum().sum()
total_low_risk = df[Low_risk_col].sum().sum()
grand_total = total_high_risk + total_low_risk

rec = {
"total_high_risk": total_high_risk,
"total_low_risk": total_low_risk,
"grand_total": grand_total,
"high_risk_rate": round(total_high_risk / grand_total * 100, 2) if grand_total else 0,
"low_risk_rate": round(total_low_risk / grand_total * 100, 2) if grand_total else 0
}


risk = pd.DataFrame([rec])

risk

#yes, arounf 56.81 of the voilations are in high risk category


,total_high_risk,total_low_risk,grand_total,high_risk_rate,low_risk_rate
0,481610,366156,847766,56.81,43.19
